# B2.11 · Context engineering for the pipeline

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.10 · Severity calibration and reporting](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**.

| | |
|---|---|
| Tools used | tree-sitter, GLM-4.6, Llama 3.3, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Compare four context strategies against one bug and measure which are decidable and at what size.

**Why a security engineer needs it.** The model is given the repository and asked to be thorough, so the relevant line falls out of the window. The control it builds is: slice on the source-sink path, not on distance: the smallest context that still supports a severity decision.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Give an agent more context and it gets better, until it gets worse. The cliff is real, it arrives earlier than anyone expects, and past it you are paying more per token for a worse answer.

> **At CyberTravels.** Give the review agent CyberTravels' whole repository and it gets worse, not better. The cliff arrives earlier than anyone expects and you pay more per token for it.

## 2 · The framework

```
   accuracy
     ^
     |          .-----.
     |        .'       `.
     |      .'           `.       <- the cliff
     |    .'               `--....
     |  .'
     +--------------------------------> context tokens
        too little        enough      too much

   past the peak you pay more per token for a worse answer
```

Cross-cutting, and it applies to every stage that calls a model: stages 3, 4, 5,
7 and 14.

The instinct when a model misses something is to give it more context. Usually
the opposite is correct.

To find a vulnerability, a model needs three things: the **sink**, the
**source**, and the **path** between them. Everything else competes for
attention and for window. A repository dumped into a prompt does not produce a
thorough review — it produces a review of whatever survived truncation, and you
cannot tell which parts those were.

So context engineering is mostly subtraction, with one exception you must not
subtract: the **enclosing signature**, because that is where reachability is
decided. The identical concatenation is critical inside an HTTP handler and
irrelevant inside a migration script that takes a constant.

## 3 · The stage, as a skill

Context engineering here is not "send less" — it is finding the slice in which the defect is decidable at all, and only then making it smaller. The skill measures four candidate slices and reports which are decidable and what each carries that the defect does not depend on.

### The skill — [`skills/appsec/context-window-sizing/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/context-window-sizing/SKILL.md)

```yaml
name: context-window-sizing
description: >-
  Find the smallest slice of a file in which a defect is actually decidable, and
  measure what larger windows add in unrelated code. Use when tuning what a
  model is shown per finding, when analysis costs too much, or when a model
  keeps missing a bug that is on the line you gave it.
allowed-tools: Read, Grep, Glob
```

# Decidable, then small — in that order

Context engineering for an analysis pipeline is not "send less". It is finding
the slice in which the question can be answered at all, and only then making it
smaller. A ±2-line window around the bug is cheap and **not decidable**: it
lacks the signature, so nothing in it says where the value came from.

## When to use this

When designing what a pipeline sends per finding, and whenever cost per finding
is the constraint.

## Procedure

**1 — Define decidability for the defect class.** For injection: the sink, the
value's origin, and any sanitiser between them. Write it down before slicing, or
you will judge slices by how they look.

**2 — Build the candidate slices.** The whole file, a fixed window around the
line, a wider window, and a **path slice** — the enclosing function plus the
definitions it depends on. Measure each in characters.

**3 — Mark each slice decidable or not,** against step 1. Cheap and undecidable
is the trap: it looks like a saving and it produces confident answers about
information that is not there.

**4 — Count unrelated content in the decidable ones.** Functions, constants and
imports the defect does not depend on. This is what a bigger window costs, in
tokens and in the model's attention.

**5 — Pick the smallest decidable slice, and say what it excluded.** The
exclusion list is what somebody re-reads when the pipeline misses something.

## Output contract

```json
{
  "decidability": {"requires": ["str"]},
  "slices": [{"name": "str", "chars": 0, "decidable": false, "unrelated_units": 0}],
  "chosen": {"name": "str", "chars": 0, "unrelated_units": 0},
  "excluded": ["str"]
}
```

## Failure modes

- **Optimising size first.** An undecidable slice is not cheap; it is wrong at a
  lower price.
- **Judging slices by eye.** Write the decidability requirement down first.
- **Ignoring unrelated content** in a decidable slice. It is the cost you can
  actually remove.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/context-window-sizing/scripts/context_window_sizing.py
SCRIPT = "skills/appsec/context-window-sizing/scripts/context_window_sizing.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The whole file is roughly 840 characters, the ±2 window about 200 and the path slice about 390. The ±2 window is not decidable because it lacks the signature; the ±6 window and the whole file are decidable but carry unrelated functions. The path slice is the smallest decidable context with zero unrelated functions, about 53% smaller than the whole file.

## Your turn

Apply the path-slice rule where the source is three functions away from the sink. That is the case where text windows break down entirely and the call graph the threat model derives (B2.2) earns its keep.

---

**Next → [B2.12 · Securing the developers' coding agents](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*